# Lab 7 — Machine Learning, and a Macro Model as a Loss Function

*ECON 282E · Session 7 · Thursday, November 5, 2026*

Session 7 stopped choosing the approximating family in advance. This lab builds the instrument and
then points it at a model.

**Part 1** a network is a basis you do not choose · **Part 2** the training loop, in five lines ·
**Part 3** backpropagation by hand, checked against `autograd` · **Part 4** the growth model as a
loss function · **Part 5** the quarterly RBC, and how wrong it is · **Part 6** shape restrictions
that hold identically.

Everything runs on a CPU in a few minutes. This is the first lab where a GPU helps, and it is still
not needed.

> Numbers quoted from the lecture come from `tools/figures/s07_numbers.json`. Where this notebook
> reproduces one, it says so — and where your run disagrees, **the notebook is the check on the
> deck, not the other way round**.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)   # 3.A1: accounting identities do not tolerate float32
np.random.seed(0); torch.manual_seed(0)

ALPHA, BETA, DELTA = 0.33, 0.99, 0.025   # quarterly, as in S4-S7
RHO, SIG_EPS = 0.95, 0.007
BM_ALPHA, BM_BETA = 0.36, 0.95           # Brock-Mirman, delta = 1, closed form available
print("torch", torch.__version__)

## Part 1 — A network is a basis you do not choose

Session 5 wrote every approximation as $\sum_i \theta_i \psi_i(x)$ with $\{\psi_i\}$ fixed in
advance. Take the simplest possible network — one hidden ReLU layer — and write it out:

$$\hat y(x) = a_0 + \sum_{i=1}^{N} a_i \,\mathrm{ReLU}(x - b_i).$$

The shifts $b_i$ are the **kinks**; the coefficients $a_i$ set the slope on each piece.

**Fix the $b_i$** and this is linear in the remaining parameters — an ordinary least-squares
regression on a piecewise-linear basis. Let us do that first.

In [ ]:
def relu_design(x, b):
    """Design matrix: a constant, then one ReLU column per shift."""
    return np.column_stack([np.ones_like(x)] + [np.maximum(0.0, x - bi) for bi in b])

x = np.linspace(-np.pi, np.pi, 801)
y = np.sin(x)

def fit_fixed(N):
    b = np.linspace(-np.pi, np.pi, N + 2)[1:-1]     # evenly spaced knots
    A = relu_design(x, b)
    a, *_ = np.linalg.lstsq(A, y, rcond=None)       # 5.B5's least squares
    return b, A @ a

for N in (2, 4, 8, 16):
    _, yhat = fit_fixed(N)
    print(f"N = {N:2d}   unknowns = {N+1:2d}   max|err| = {np.abs(yhat - y).max():.4f}")

Doubling $N$ roughly halves the error — the $O(h)$ rate of a linear element, exactly as 5.B4 would
predict. **There is no network here yet.**

Now let the shifts be trained too. That single change turns the regression into a one-hidden-layer
ReLU network.

In [ ]:
def fit_learned(N, target=None, lo=-np.pi, hi=np.pi, iters=4000, seed=0):
    """Same functional form; now b is a parameter as well."""
    torch.manual_seed(seed)
    xt = torch.tensor(x if target is None else target[0]).view(-1, 1)
    yt = torch.tensor(y if target is None else target[1]).view(-1, 1)
    b  = torch.tensor(np.linspace(lo, hi, N + 2)[1:-1], requires_grad=True)
    a  = torch.zeros(N, requires_grad=True)
    a0 = torch.zeros(1, requires_grad=True)
    opt = torch.optim.Adam([a, b, a0], lr=0.05)
    for _ in range(iters):
        opt.zero_grad()
        pred = a0 + (torch.relu(xt - b) * a).sum(1, keepdim=True)
        ((pred - yt) ** 2).mean().backward()
        opt.step()
    with torch.no_grad():
        pred = (a0 + (torch.relu(xt - b) * a).sum(1, keepdim=True)).numpy().ravel()
    return b.detach().numpy(), pred

print(f"{'N':>3} {'fixed knots':>12} {'learned knots':>14}")
for N in (2, 4, 8, 16):
    _, yf = fit_fixed(N)
    _, yl = fit_learned(N)
    print(f"{N:>3} {np.abs(yf-y).max():>12.4f} {np.abs(yl-y).max():>14.4f}")

**Exercise 1.** At $N=8$ the learned version should be about ten times better. At $N=4$ it is
*worse*. Explain the second fact in one sentence, and name the frame in 7.A2 that predicts it.

*(Hint: least squares is a solved problem. Training is not.)*

### Where it really matters: a kink

A borrowing constraint puts a kink in the consumption function. A fixed, evenly spaced basis has to
be lucky to have a knot there; a trained one can go and find it.

In [ ]:
xk = np.linspace(0.0, 4.0, 801)
yk = np.minimum(xk, 1.0 + 0.25 * (xk - 1.0))       # kink exactly at x = 1

print(f"{'N':>3} {'fixed':>10} {'learned':>10}   nearest learned knot to the true kink")
for N in (2, 4, 8):
    bfix = np.linspace(0.0, 4.0, N + 2)[1:-1]
    A = relu_design(xk, bfix); c, *_ = np.linalg.lstsq(A, yk, rcond=None)
    ef = np.abs(A @ c - yk).max()
    bl, yl = fit_learned(N, target=(xk, yk), lo=0.0, hi=4.0)
    el = np.abs(yl - yk).max()
    near = min(bl, key=lambda v: abs(v - 1.0))
    print(f"{N:>3} {ef:>10.4f} {el:>10.4f}   {near:.4f}")

With only **two** kinks to spend, training puts one essentially exactly at $x=1$. Nobody told it
where the constraint binds. That is the whole of 7.A2 in one number, and it is the reason the rest
of this session is worth the trouble.

**Exercise 2.** Fit a degree-16 Chebyshev polynomial to the same kinked target and compare the
maximum error. You should see 5.B2's Gibbs ringing across the whole domain.

## Part 2 — The training loop, in five lines

Everything from here on is the same five steps: **zero, forward, score, backward, step**.

In [ ]:
net = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 1))
opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)   # course default

xt = torch.tensor(x).view(-1, 1)
yt = torch.tensor(y).view(-1, 1)

for it in range(3000):
    opt.zero_grad()                    # 1. zero  -- the line people forget
    pred = net(xt)                     # 2. forward
    loss = ((pred - yt) ** 2).mean()   # 3. score
    loss.backward()                    # 4. backward
    opt.step()                         # 5. step

print("final loss", float(loss.detach()))
print("parameters", sum(p.numel() for p in net.parameters()))

## Part 3 — Backpropagation by hand, checked against `autograd`

The worked example from 7.A3. One input, one sigmoid hidden unit, one sigmoid output unit:

$$z_1 = w_1x+b_1,\quad h=\sigma(z_1),\quad z_2=w_2h+b_2,\quad \hat y=\sigma(z_2),\quad
L=\tfrac12(\hat y-t)^2.$$

Do it by hand first. Then let the tape do it, and check they agree.

In [ ]:
import math
sig = lambda v: 1.0 / (1.0 + math.exp(-v))

xv, t = 1.0, 0.0
w1, b1, w2, b2, eta = 0.5, 0.0, 2.0, 0.0, 0.1

z1 = w1 * xv + b1;  h  = sig(z1)
z2 = w2 * h  + b2;  yh = sig(z2)
L  = 0.5 * (yh - t) ** 2
print(f"forward : z1={z1:.4f}  h={h:.4f}  z2={z2:.4f}  yhat={yh:.4f}  L={L:.4f}")

d2     = (yh - t) * yh * (1 - yh)          # dL/dz2, the reusable error signal
dL_dw2 = d2 * h
dL_dh  = d2 * w2
d1     = dL_dh * h * (1 - h)               # dL/dz1
dL_dw1 = d1 * xv
print(f"backward: delta2={d2:.4f}  dL/dw2={dL_dw2:.4f}  dL/dw1={dL_dw1:.4f}")
print(f"update  : w2 -> {w2 - eta*dL_dw2:.4f}   w1 -> {w1 - eta*dL_dw1:.4f}")

In [ ]:
tw1 = torch.tensor(w1, requires_grad=True); tb1 = torch.tensor(b1, requires_grad=True)
tw2 = torch.tensor(w2, requires_grad=True); tb2 = torch.tensor(b2, requires_grad=True)
th  = torch.sigmoid(tw1 * xv + tb1)
ty  = torch.sigmoid(tw2 * th + tb2)
(0.5 * (ty - t) ** 2).backward()

print("autograd dL/dw1 =", float(tw1.grad), " by hand:", dL_dw1)
print("autograd dL/dw2 =", float(tw2.grad), " by hand:", dL_dw2)
assert abs(float(tw1.grad) - dL_dw1) < 1e-12 and abs(float(tw2.grad) - dL_dw2) < 1e-12
print("they agree to machine precision")

**Exercise 3.** Both $\hat y(1-\hat y)$ and $h(1-h)$ are at most $0.25$. Stack $L$ sigmoid layers
and the error signal reaching the first one is multiplied by at most $0.25^L$. Compute that for
$L=1,\dots,7$ and say at what depth the first layer has effectively stopped learning.

**Exercise 4.** Replace the hidden sigmoid with a ReLU and redo the hand calculation. Which factor
disappears, and why is that most of the reason ReLU displaced the sigmoid?

## Part 4 — A macro model as a loss function

No data from here on. The **model's own Euler equation** supplies the target: the label attached to
every sampled state is zero.

Start with the case where the answer is known. Brock–Mirman with log utility and $\delta=1$ has the
closed-form policy $k' = \alpha\beta k^{\alpha}$ — 1.B1's benchmark. *If a method cannot recover
this, do not trust it on anything harder.*

In [ ]:
a, b = BM_ALPHA, BM_BETA
kss   = (a * b) ** (1.0 / (1.0 - a))
klo, khi = 0.5 * kss, 1.5 * kss
print(f"Brock-Mirman steady state k* = {kss:.6f}   band = [{klo:.4f}, {khi:.4f}]")

class Policy(nn.Module):
    """c(k) = sigmoid(net) * output, so 0 < c < k^alpha IDENTICALLY (7.B1)."""
    def __init__(self, hidden=32):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(1, hidden), nn.Tanh(),
                               nn.Linear(hidden, hidden), nn.Tanh(),
                               nn.Linear(hidden, 1))
    def forward(self, k):
        kn = 2.0 * (k - klo) / (khi - klo) - 1.0      # normalise the input
        return torch.sigmoid(self.f(kn))

pol = Policy()
opt = torch.optim.Adam(pol.parameters(), lr=3e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=25000, eta_min=1e-6)

for it in range(25000):
    k  = torch.rand(512, 1) * (khi - klo) + klo       # a FRESH batch of states each step
    y  = k ** a
    c  = pol(k) * y
    kp = torch.clamp(y - c, klo, khi)
    cp = pol(kp) * kp ** a
    resid = 1.0 - (c / cp) * b * a * kp ** (a - 1.0)  # unit-free Euler residual
    loss = (resid ** 2).mean()
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    if (it + 1) % 5000 == 0:
        print(f"  iter {it+1:5d}   loss {float(loss.detach()):.3e}")

In [ ]:
# Grade it against the closed form -- the only rung of the hierarchy that tests CORRECTNESS.
kt = np.linspace(klo, khi, 400)
with torch.no_grad():
    c_net = (pol(torch.tensor(kt).view(-1, 1)).numpy().ravel()) * kt ** a
kp_net = kt ** a - c_net
kp_true = a * b * kt ** a

print(f"max |k'_net - k'_true|   = {np.abs(kp_net - kp_true).max():.3e}")
rel = np.abs(kp_net / kp_true - 1.0)
print(f"max  relative error      = {rel.max():.3e}")
print(f"median relative error    = {np.median(rel):.3e}")
print(f"implied saving rate      = {np.median(kp_net/kt**a):.4f}   (true alpha*beta = {a*b:.4f})")
print(f"worst k                  = {kt[np.argmax(rel)]:.4f}   "
      f"(band is [{klo:.4f}, {khi:.4f}])")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
ax[0].plot(kt, kp_net - kt, label="network"); ax[0].plot(kt, kp_true - kt, "--", label="closed form")
ax[0].axhline(0, color="0.7", lw=0.8); ax[0].set_xlabel("k"); ax[0].set_ylabel("k' - k")
ax[0].set_title("saving (deviation, not level)"); ax[0].legend()
ax[1].plot(kt, np.log10(np.abs(kp_net/kp_true - 1) + 1e-16))
ax[1].set_xlabel("k"); ax[1].set_ylabel("log10 relative error"); ax[1].set_title("where the error lives")
plt.tight_layout(); plt.show()

**Exercise 5.** The *median* relative error is tiny and the implied saving rate is $lphaeta$ to
four decimals — the network has solved the model. The *maximum* is enormous, and the cell prints
where it is. Look at the right-hand panel and say what is happening there. Why? (7.B1 measured the same thing on the quarterly model: two decades of accuracy lost
between $k^*$ and $0.5k^*$.)

**Exercise 6.** Retrain sampling $k$ from $[0.4k^*, 1.6k^*]$ but evaluate on $[0.5k^*, 1.5k^*]$.
State the rule this suggests in one line.

**Exercise 7.** Plot the level $k'$ against $k$ instead of $k'-k$. Convince yourself that it shows
nothing, and note the house rule: *plot the deviation, not the level*.

## Part 5 — The quarterly RBC, and how wrong it is

Now the model 3.A4, 4.B5, 5.A6 and 5.B6 all solved, with a seven-state Rouwenhorst chain. There is
no closed form, so we grade on **off-sample Euler residuals** against Judd's (1998) scale:

| $\log_{10}|\mathcal{R}|$ | verdict |
|---|---|
| $< -3$ | acceptable |
| $< -5$ | good |
| $< -7$ | excellent |

In [ ]:
def rouwenhorst(n, rho, sigma):
    """Copied from tools/figures/s04_generate.py so every ledger uses one chain."""
    p = (1.0 + rho) / 2.0
    P = np.array([[p, 1 - p], [1 - p, p]])
    for k in range(3, n + 1):
        Z = np.zeros((k, k))
        Z[:-1, :-1] += p * P; Z[:-1, 1:] += (1 - p) * P
        Z[1:, :-1] += (1 - p) * P; Z[1:, 1:] += p * P
        Z[1:-1, :] /= 2.0
        P = Z
    sz = sigma / np.sqrt(1.0 - rho ** 2)
    psi = sz * np.sqrt(n - 1)
    return np.linspace(-psi, psi, n), P

logz, Pi = rouwenhorst(7, RHO, SIG_EPS)
zs = np.exp(logz); nz = len(zs)
kss_q = ((1.0 / BETA - 1.0 + DELTA) / ALPHA) ** (1.0 / (ALPHA - 1.0))
qlo, qhi = 0.5 * kss_q, 1.5 * kss_q
print(f"quarterly k* = {kss_q:.4f}   band = [{qlo:.3f}, {qhi:.3f}]   7 shock states")

In [ ]:
class PolicyZ(nn.Module):
    """c(k, z) with feasibility built in; the shock enters one-hot."""
    def __init__(self, hidden=64, layers=3):
        super().__init__()
        seq, prev = [], 1 + nz
        for _ in range(layers):
            seq += [nn.Linear(prev, hidden), nn.Tanh()]; prev = hidden
        seq += [nn.Linear(prev, 1)]
        self.f = nn.Sequential(*seq)
    def forward(self, kn, zoh):
        return torch.sigmoid(self.f(torch.cat([kn, zoh], 1)))

zt, Pit = torch.tensor(zs), torch.tensor(Pi)
eye = torch.eye(nz)
norm = lambda k: 2.0 * (k - qlo) / (qhi - qlo) - 1.0

torch.manual_seed(0)
netq = PolicyZ()
optq = torch.optim.Adam(netq.parameters(), lr=3e-3)
schq = torch.optim.lr_scheduler.CosineAnnealingLR(optq, T_max=12000, eta_min=1e-5)

for it in range(12000):
    k = torch.rand(1024, 1) * (qhi - qlo) + qlo
    j = torch.randint(0, nz, (1024,))
    y = zt[j].view(-1, 1) * k ** ALPHA + (1 - DELTA) * k
    c = netq(norm(k), eye[j]) * y
    kp = torch.clamp(y - c, qlo, qhi)
    rhs = torch.zeros_like(c)
    for l in range(nz):                                  # EXACT sum: never sample what you can sum
        cp = netq(norm(kp), eye[l].expand(1024, nz)) * (zt[l] * kp ** ALPHA + (1 - DELTA) * kp)
        rhs = rhs + Pit[j, l].view(-1, 1) * (ALPHA * zt[l] * kp ** (ALPHA - 1) + (1 - DELTA)) / cp
    loss = ((1.0 - c * BETA * rhs) ** 2).mean()
    optq.zero_grad(); loss.backward(); optq.step(); schq.step()
    if (it + 1) % 4000 == 0:
        print(f"  iter {it+1:5d}   loss {float(loss.detach()):.3e}")

In [ ]:
def cfun(kv, j):
    with torch.no_grad():
        kt_ = torch.tensor(np.atleast_1d(kv)).view(-1, 1)
        y = zs[j] * kt_ ** ALPHA + (1 - DELTA) * kt_
        return (netq(norm(kt_), eye[j].expand(kt_.shape[0], nz)) * y).numpy().ravel()

def euler_errors(band, n_test=2000, seed=0):
    rng = np.random.default_rng(seed)
    kt_ = rng.uniform(band[0], band[1], n_test)
    jt  = rng.integers(0, nz, n_test)
    out = []
    for i in range(n_test):
        j = int(jt[i]); c = float(cfun(np.array([kt_[i]]), j)[0])
        y = zs[j] * kt_[i] ** ALPHA + (1 - DELTA) * kt_[i]
        kp = min(max(y - c, qlo), qhi)
        rhs = sum(Pi[j, l] * (ALPHA * zs[l] * kp ** (ALPHA - 1) + (1 - DELTA))
                  / max(float(cfun(np.array([kp]), l)[0]), 1e-12) for l in range(nz))
        out.append(abs(1.0 - c * BETA * rhs))
    return np.array(out)

wide = euler_errors((qlo, qhi))
near = euler_errors((0.9 * kss_q, 1.1 * kss_q))
span = qhi - qlo
inner = euler_errors((qlo + 0.05 * span, qhi - 0.05 * span))
print(f"max log10 |R|, whole band   = {np.log10(wide.max()):+.2f}")
print(f"max log10 |R|, interior 90% = {np.log10(inner.max()):+.2f}")
print(f"max log10 |R|, near k*      = {np.log10(near.max()):+.2f}")
print(f"\nFor comparison, 5.B6's Chebyshev collocation on THIS MODEL: -6.40 across the band,")
print( "with 56 stored numbers and 0.13 seconds.")

**Exercise 8.** Your network has thousands of parameters, took minutes, and does not reach Judd's
*acceptable* across the band. Chebyshev reaches nearly *excellent* with 56 numbers in a tenth of a
second. **Is the network implemented badly, or is this the right answer?** Defend your view in
three sentences, using 7.A2's Theorem 6.

**Exercise 9.** Every column of that comparison scales differently with the number of state
variables. Write down how each one scales, and say roughly where the crossover is.

## Part 6 — Structure that holds identically

Theory says a value function is concave. A generic network does not know that. Fit one to a function
that is *exactly* convex and look at what comes out.

In [ ]:
xs = np.linspace(-1, 1, 401)
ys = 0.6 * xs ** 2 + 0.25 * xs + 0.1          # second derivative is 1.2 everywhere

def convexity_violation(xv, yv):
    """Largest negative second difference; zero means convex."""
    h = xv[1] - xv[0]
    d2 = (yv[2:] - 2 * yv[1:-1] + yv[:-2]) / h ** 2
    return max(0.0, -d2.min())

torch.manual_seed(0)
ff = nn.Sequential(nn.Linear(1, 32), nn.ReLU(), nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 1))
o  = torch.optim.Adam(ff.parameters(), lr=0.01)
xtt, ytt = torch.tensor(xs).view(-1, 1), torch.tensor(ys).view(-1, 1)
for _ in range(4000):
    o.zero_grad(); ((ff(xtt) - ytt) ** 2).mean().backward(); o.step()
with torch.no_grad():
    yff = ff(xtt).numpy().ravel()

print(f"plain network: RMSE = {np.sqrt(((yff-ys)**2).mean()):.5f}"
      f"   worst -f'' = {convexity_violation(xs, yff):.2f}   (true curvature is 1.2)")

The fit looks acceptable and the function is **locally concave in places**. In a Bellman solver that
is not untidy, it is fatal: the inner maximisation acquires local maxima and the policy jumps.

An **input convex neural network** (Amos, Xu & Kolter 2017) cannot do this. Keep the recurrent
weights non-negative and use a convex, non-decreasing activation, and convexity is an identity.

In [ ]:
class ICNN(nn.Module):
    """Non-negative recurrent weights (via softplus, NOT clamp) + ReLU => convex in x."""
    def __init__(self, hidden=32, layers=2):
        super().__init__()
        self.Wy = nn.ModuleList([nn.Linear(1, hidden) for _ in range(layers)] + [nn.Linear(1, 1)])
        self.Wz = nn.ModuleList([nn.Linear(hidden, hidden, bias=False) for _ in range(layers - 1)]
                                + [nn.Linear(hidden, 1, bias=False)])
    def forward(self, xv):
        z = torch.relu(self.Wy[0](xv))
        for i, W in enumerate(self.Wz):
            z = torch.nn.functional.linear(z, torch.nn.functional.softplus(W.weight)) + self.Wy[i+1](xv)
            if i < len(self.Wz) - 1:
                z = torch.relu(z)
        return z

torch.manual_seed(0)
ic = ICNN(); o2 = torch.optim.Adam(ic.parameters(), lr=0.01)
for _ in range(4000):
    o2.zero_grad(); ((ic(xtt) - ytt) ** 2).mean().backward(); o2.step()
with torch.no_grad():
    yic = ic(xtt).numpy().ravel()
print(f"ICNN         : RMSE = {np.sqrt(((yic-ys)**2).mean()):.5f}"
      f"   worst -f'' = {convexity_violation(xs, yic):.2e}")

**Exercise 10.** Point the same ICNN at $\sin 3x$, which is not convex. It will fail — and it will
fail *while remaining exactly convex*. Explain why that failure is more useful than a silently wrong
fit.

**Exercise 11.** The implementation uses `softplus` on the weights rather than `clamp`. Replace it
with `clamp(min=0)` and watch what happens to the weights that hit the boundary. Which frame in 7.A2
is this?

## What to take away

1. **Fix the knots and a one-layer ReLU network is least squares on a piecewise-linear basis.** Let
   them move and it finds the kink. That is the entire difference Session 7 is about.
2. **Training is not least squares.** At $N=4$ the learned version was worse. Universal
   approximation is existence, not a recipe.
3. **An economic model supplies its own labels** — zero, at every sampled state, with an unlimited
   supply of states.
4. **Feasibility belongs in the parameterisation**, not in a clip: $c=\sigma(\cdot)\times$ cash.
5. **Never sample what you can sum.** The inner expectation over a finite Markov chain is exact and
   costs one matrix product.
6. **A sampled method is weakest at the edge of the region it sampled.** Sample wider than you
   evaluate.
7. **On a smooth, one-dimensional model the network loses, decisively.** Report the number anyway.
   The case for the method is how its cost scales, not how it does here.
8. **Impose the shape theory gives you.** An ICNN is convex identically — and here it was more
   accurate too, because the constraint removed wrong functions from the search.